# Data-center census — all sites in one notebook

Runs the ring analysis for every site in `census/census_sites.csv`. Safe to re-run: sites that are
already exported or already summarized are skipped.

**Order:** Part 0 once per session → Part 1 (starts Earth Engine exports; they run on Google's servers,
10–60 min) → Part 2 until every export shows COMPLETED → Part 3 (labels) → Part 4 (summaries) →
Part 5 (checks). Results are written to `2026/Remote-Sensing/census/outputs/` on Drive.

## Part 0 — Setup

In [ ]:
EE_PROJECT = ""   # <- your Earth Engine Cloud project id (the same one your site notebooks use)

import ee
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

from google.colab import drive
drive.mount("/content/drive")

import os, sys, importlib
import numpy as np, pandas as pd

BASE = "/content/drive/MyDrive/2026/Remote-Sensing/census"
sys.path.insert(0, f"{BASE}/code")
import dc_heat.stats, dc_heat.windows, dc_heat.summary
for module in (dc_heat.stats, dc_heat.windows, dc_heat.summary):
    importlib.reload(module)
from dc_heat import stats, windows, summary

SITES = pd.read_csv(f"{BASE}/census_sites.csv", parse_dates=["operation_start"])
REQUIRED = ["site_id", "role", "name", "group", "lat", "lon", "construction_year", "construction_source",
            "announced_year", "operation_start", "paper_base_years", "paper_op_years"]
missing = [c for c in REQUIRED if c not in SITES.columns]
if missing:
    raise ValueError(f"census_sites.csv is missing columns {missing} — it must be the file Claude generated, "
                     "not the verification checklist")
EXPORT_FOLDER = "dc_heat_census_exports"            # Earth Engine writes here, at the top of My Drive
EXPORT_DIR = f"{BASE}/outputs/{EXPORT_FOLDER}"     # the exports folder was moved under census/outputs
OUT = f"{BASE}/outputs"
os.makedirs(OUT, exist_ok=True)
REDO = ["EPOCH-05"]   # sites whose coordinates changed: Parts 1 and 3 rebuild their export and labels
print(len(SITES), "rows:", SITES.role.value_counts().to_dict(), "| census groups:",
      SITES[SITES.role == "census"].group.value_counts().to_dict())

## Part 1 — Start one Earth Engine export per site

Each export writes every clear Landsat 8/9 scene, 2013–2026, for the five rings: LST (°C), NDVI,
emissivity and albedo (Liang 2001). Same preprocessing as Section 10 of the site notebooks, plus albedo.

In [ ]:
RINGS = [(0.0, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 3.5), (3.5, 5.0)]   # km
START, END = "2013-01-01", "2026-12-31"

def prep(img):
    lst = img.select("ST_B10").multiply(0.00341802).add(149.0).subtract(273.15).rename("LST_C")
    sr = img.select(["SR_B2", "SR_B4", "SR_B5", "SR_B6", "SR_B7"]).multiply(0.0000275).add(-0.2)
    ndvi = sr.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")
    albedo = sr.expression("0.356*B2 + 0.130*B4 + 0.373*B5 + 0.085*B6 + 0.072*B7 - 0.0018",
                           {b: sr.select(f"SR_{b}") for b in ["B2", "B4", "B5", "B6", "B7"]}).rename("ALBEDO")
    emis = img.select("ST_EMIS").multiply(0.0001).rename("EMIS")
    qa = img.select("QA_PIXEL")
    clear = qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0))
    return (lst.addBands([ndvi, emis, albedo]).updateMask(clear)
               .copyProperties(img, ["system:time_start"]))

def ring_fc(lon, lat, site_id):
    pt = ee.Geometry.Point(lon, lat)
    feats = []
    for i, (r0, r1) in enumerate(RINGS):
        outer = pt.buffer(r1 * 1000)
        geom = outer if r0 == 0 else outer.difference(pt.buffer(r0 * 1000))
        feats.append(ee.Feature(geom, {"ring": i, "site_id": site_id}))
    return ee.FeatureCollection(feats)

def ring_scenes(lon, lat, site_id):
    fc = ring_fc(lon, lat, site_id)
    col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
             .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
             .filterBounds(fc).filterDate(START, END).map(prep))
    def per_image(img):
        date = img.date().format("YYYY-MM-dd")
        return (img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=30, tileScale=4)
                   .map(lambda f: f.set("date", date)))
    return col.map(per_image).flatten().filter(ee.Filter.notNull(["LST_C"]))

active = {t.config["description"] for t in ee.batch.Task.list() if t.state in ("READY", "RUNNING")}
started = 0
for s in SITES.itertuples():
    name = f"rings_scenes_{s.site_id}"
    path = f"{EXPORT_DIR}/{name}.csv"
    if s.site_id in REDO and os.path.exists(path) and not os.path.exists(f"{path}.old"):
        os.rename(path, f"{path}.old")   # set the old-coordinate export aside, once
        print(f"{s.site_id}: old export renamed to {name}.csv.old; re-exporting")
    if os.path.exists(path) or name in active:
        continue
    task = ee.batch.Export.table.toDrive(
        collection=ring_scenes(s.lon, s.lat, s.site_id), description=name, folder=EXPORT_FOLDER,
        fileNamePrefix=name, fileFormat="CSV",
        selectors=["site_id", "date", "ring", "LST_C", "NDVI", "EMIS", "ALBEDO"])
    task.start()
    started += 1
print(f"started {started} exports; {len(SITES) - started} already done or running")

## Part 2 — Check export progress (re-run until all are COMPLETED)

In [ ]:
tasks = [t for t in ee.batch.Task.list() if t.config["description"].startswith("rings_scenes_")]
latest = {}
for t in tasks:                      # Task.list() is newest first; keep the latest per site
    latest.setdefault(t.config["description"], t)
counts = pd.Series([t.state for t in latest.values()]).value_counts()
print(counts.to_string())
for d, t in sorted(latest.items()):
    if t.state == "FAILED":
        print("FAILED", d, t.status().get("error_message", ""))
print("\nfiles visible on Drive:", len([f for f in os.listdir(EXPORT_DIR) if f.endswith(".csv")]) if os.path.isdir(EXPORT_DIR) else 0)
print("(if COMPLETED files are not visible yet, run: drive.flush_and_unmount(); drive.mount('/content/drive', force_remount=True))")

## Part 3 — Land cover and climate labels (runs directly, about 1–2 min)

NLCD class fractions in the 0–0.5 km core for every NLCD year (so the pre-construction year can be
chosen later), and a dryness index from TerraClimate (precipitation ÷ potential evapotranspiration,
2013–2022; below about 0.5 is dry).

In [ ]:
LABELS = f"{OUT}/site_labels.csv"
existing_labels = pd.read_csv(LABELS) if os.path.exists(LABELS) else pd.DataFrame(columns=["site_id"])
existing_labels = existing_labels[~existing_labels.site_id.isin(REDO)]   # always rebuilt
todo = SITES[~SITES.site_id.isin(existing_labels.site_id)]
if todo.empty:
    print("site_labels.csv already covers every site")
else:
    cores = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point(s.lon, s.lat).buffer(500), {"site_id": s.site_id})
        for s in todo.itertuples()])
    nlcd = ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD").merge(
           ee.ImageCollection("USGS/NLCD_RELEASES/2021_REL/NLCD"))
    years = nlcd.aggregate_array("system:index").getInfo()
    rows = []
    for idx in years:
        img = nlcd.filter(ee.Filter.eq("system:index", idx)).first().select("landcover")
        fc = img.reduceRegions(cores, ee.Reducer.frequencyHistogram(), scale=30).getInfo()["features"]
        for f in fc:
            hist = f["properties"].get("histogram") or {}
            total = sum(hist.values()) or 1
            row = {"site_id": f["properties"]["site_id"], "nlcd_year": int(str(idx)[-4:])}
            row.update({f"nlcd_{k}": v / total for k, v in hist.items()})
            rows.append(row)
    labels = pd.DataFrame(rows).fillna(0)

    tc = ee.ImageCollection("IDAHO_EPSCOR/TERRACLIMATE").filterDate("2013-01-01", "2023-01-01")
    clim = ee.Image.cat(tc.select("pr").mean(), tc.select("pet").mean().multiply(0.1))
    cfc = clim.reduceRegions(cores, ee.Reducer.mean(), scale=4638).getInfo()["features"]
    aridity = pd.DataFrame([{"site_id": f["properties"]["site_id"],
                             "aridity": f["properties"]["pr"] / f["properties"]["pet"]} for f in cfc])
    labels = labels.merge(aridity, on="site_id")
    labels = pd.concat([existing_labels, labels], ignore_index=True).fillna(0)
    labels.to_csv(LABELS, index=False)
    print(f"added labels for {todo.site_id.nunique()} sites; file now covers {labels.site_id.nunique()} sites")

## Part 4 — Summarize every site

For each site: construction year (from the registry; otherwise the earlier of the Landsat-detected year and the
public announcement year), baseline =
up to 5 years before construction, after = 2025–2026 (conversions: years from operation onward), then
the ring summary. The detected year is computed for every site so it can be checked against Epoch.

In [ ]:
rows, window_rows = [], []
for s in SITES[SITES.role.isin(["census", "warehouse"])].itertuples():
    path = f"{EXPORT_DIR}/rings_scenes_{s.site_id}.csv"
    if not os.path.exists(path):
        window_rows.append({"site_id": s.site_id, "reason": "export not found"})
        continue
    scenes = pd.read_csv(path, parse_dates=["date"]).rename(columns=str.lower)
    detected = windows.detect_construction_year(scenes)
    year, how = windows.resolve_construction_year(s.construction_year, detected, s.announced_year)
    w = windows.census_windows(s.group, year, s.operation_start)
    window_rows.append({"site_id": s.site_id, "construction_year": year,
                        "construction_source": s.construction_source if how == "registry" else how,
                        "detected_year": detected, "base_years": w["base_years"], "op_years": w["op_years"],
                        "n_scenes": scenes.date.nunique(), "reason": w["reason"]})
    if w["reason"]:
        continue
    site_summary = summary.summarize_site(scenes, w["base_years"], w["op_years"])
    site_summary.insert(0, "site_id", s.site_id)
    rows.append(site_summary)

census = pd.concat(rows, ignore_index=True)
census.to_csv(f"{OUT}/rings_summary.csv", index=False)
windows_used = pd.DataFrame(window_rows)
windows_used.to_csv(f"{OUT}/census_windows.csv", index=False)
print(f"summarized {census.site_id.nunique()} sites; skipped:")
print(windows_used[windows_used.reason.fillna("") != ""][["site_id", "reason"]].to_string(index=False))

## Part 5 — Checks

In [ ]:
# (a) Validation rows: the paper's points and windows must reproduce the paper's core values (within ~0.05 °C).
#     VALID-MIDLOTHIAN is the paper's point, which turned out to be a warehouse, not Google.
PAPER = {"VALID-SOCIAL-CIRCLE": 4.75, "VALID-GALLATIN": 0.49, "VALID-MIDLOTHIAN": -1.01, "VALID-MESA": -2.55}
def years(text):
    a, b = str(text).split("-") if "-" in str(text) else (text, text)
    return list(range(int(a), int(b) + 1))
for s in SITES[SITES.role == "validation"].itertuples():
    scenes = pd.read_csv(f"{EXPORT_DIR}/rings_scenes_{s.site_id}.csv", parse_dates=["date"]).rename(columns=str.lower)
    core = summary.summarize_site(scenes, years(s.paper_base_years), years(s.paper_op_years)).set_index("ring").loc[0]
    print(f"{s.site_id:20s} paper {PAPER[s.site_id]:+.2f}  now {core.dlst:+.2f}  CI [{core.ci_lo:+.2f}, {core.ci_hi:+.2f}]")

# (b) Detected construction year vs Epoch, where Epoch has one
check = windows_used.merge(SITES[["site_id", "construction_source"]], on="site_id", suffixes=("", "_reg"))
check = check[check.construction_source_reg == "epoch"]
print("\ndetected minus Epoch construction year:")
print((check.detected_year - check.construction_year).value_counts(dropna=False).sort_index().to_string())

In [ ]:
# (c) One line per site: the core (0–0.5 km) result
core = census[census.ring == 0].merge(SITES[["site_id", "name", "group"]], on="site_id")
core = core.merge(windows_used[["site_id", "construction_year", "construction_source"]], on="site_id")
cols = ["name", "group", "construction_year", "construction_source", "dlst", "ci_lo", "ci_hi", "p", "dndvi", "dalbedo", "n_base", "n_op"]
print(core.sort_values(["group", "dlst"])[cols].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Part 6 — Marinoni re-test (Phase 4)

Marinoni et al. compare each facility's monthly, deseasonalized MODIS LST in a 0–10 km disk for the 12
months after operations start with the previous 60 months, with no spatial reference. 6a picks 10
placebo points per site (20–60 km away, same pre-construction land cover, ≥10 km from any census site);
6b exports monthly MODIS (Terra MOD11A1, day and night) for the disk and a 10–20 km background ring
around every site and placebo; 6c computes their metric, a background-referenced version, and the
placebo distribution.

In [ ]:
# Part 6a — placebo points (runs directly, a few minutes; skipped if placebos.csv exists)
import dc_heat.marinoni, dc_heat.crosssite
importlib.reload(dc_heat.marinoni); importlib.reload(dc_heat.crosssite)
from dc_heat import marinoni, crosssite

windows_used = pd.read_csv(f"{OUT}/census_windows.csv")
years = dict(zip(windows_used.site_id, windows_used.construction_year))
labels = pd.read_csv(f"{OUT}/site_labels.csv").fillna(0.0)
cover = crosssite.precon_cover(labels[labels.site_id.isin(years)], years)
census_sites = (SITES[SITES.role == "census"]
                .merge(cover[["site_id", "dominant_cover"]], on="site_id")
                .rename(columns={"dominant_cover": "cover"}))
CODE_TO_GROUP = {code: group for group, codes in crosssite.COVER_GROUPS.items() for code in codes}

if os.path.exists(f"{OUT}/placebos.csv"):
    placebos = pd.read_csv(f"{OUT}/placebos.csv")
    print("placebos.csv already exists — delete it to redraw")
else:
    nlcd2019 = (ee.ImageCollection("USGS/NLCD_RELEASES/2019_REL/NLCD")
                  .filter(ee.Filter.eq("system:index", "2019")).first().select("landcover"))
    candidate_rows = []
    for s in census_sites.itertuples():
        pt = ee.Geometry.Point(s.lon, s.lat)
        ring = pt.buffer(60000, 100).difference(pt.buffer(20000, 100), 100)
        pts = (ee.FeatureCollection.randomPoints(region=ring, points=60, seed=42)
                 .map(lambda f: f.set({"lon": f.geometry().coordinates().get(0),
                                       "lat": f.geometry().coordinates().get(1)}).buffer(500)))
        for i, f in enumerate(nlcd2019.reduceRegions(pts, ee.Reducer.mode(), scale=30).getInfo()["features"]):
            props = f["properties"]
            candidate_rows.append({"site_id": s.site_id, "point_id": f"{s.site_id}-P{i:02d}",
                                   "lat": props["lat"], "lon": props["lon"],
                                   "cover": CODE_TO_GROUP.get(int(props.get("mode") or 0), "other")})
    placebos = marinoni.select_placebos(pd.DataFrame(candidate_rows), census_sites[["site_id", "lat", "lon", "cover"]],
                                        n=10, min_km=10)
    placebos.to_csv(f"{OUT}/placebos.csv", index=False)
print(f"{len(placebos)} placebos for {placebos.site_id.nunique()} of {len(census_sites)} sites")
print(placebos.groupby("site_id").size().describe()[["min", "mean", "max"]].round(1).to_dict())

In [ ]:
# Part 6b — start the monthly MODIS export (one Earth Engine task, ~30–90 min); re-run to see its status
START_M, N_MONTHS = "2004-01-01", 272          # Jan 2004 – Aug 2026
TASK = "modis_monthly"
points = ([(s.site_id, s.site_id, "site", s.lon, s.lat) for s in census_sites.itertuples()] +
          [(p.site_id, p.point_id, "placebo", p.lon, p.lat) for p in placebos.itertuples()])
features = []
for site_id, point_id, kind, lon, lat in points:
    pt = ee.Geometry.Point(lon, lat)
    props = {"site_id": site_id, "point_id": point_id, "kind": kind}
    features.append(ee.Feature(pt.buffer(10000, 100), {**props, "zone": "disk"}))
    features.append(ee.Feature(pt.buffer(20000, 100).difference(pt.buffer(10000, 100), 100), {**props, "zone": "annulus"}))
zones = ee.FeatureCollection(features)

modis = ee.ImageCollection("MODIS/061/MOD11A1").select(["LST_Day_1km", "LST_Night_1km"])
def monthly(i):
    start = ee.Date(START_M).advance(i, "month")
    img = modis.filterDate(start, start.advance(1, "month")).mean().multiply(0.02).subtract(273.15)
    return img.rename(["day", "night"]).set("month", start.format("YYYY-MM"))
def per_month(img):
    return (img.reduceRegions(collection=zones, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
               .map(lambda f: f.set("month", img.get("month"))))
table = ee.ImageCollection.fromImages(ee.List.sequence(0, N_MONTHS - 1).map(monthly)).map(per_month).flatten()

existing = [t for t in ee.batch.Task.list() if t.config["description"] == TASK]
if os.path.exists(f"{EXPORT_DIR}/{TASK}.csv"):
    print(f"{TASK}.csv is on Drive — run 6c")
elif existing and existing[0].state in ("READY", "RUNNING", "COMPLETED"):
    print(f"{TASK}: {existing[0].state}")
else:
    ee.batch.Export.table.toDrive(collection=table, description=TASK, folder=EXPORT_FOLDER, fileNamePrefix=TASK,
                                  fileFormat="CSV", selectors=["site_id", "point_id", "kind", "zone", "month", "day", "night"]).start()
    print(f"started {TASK}: {len(points)} points x 2 zones x {N_MONTHS} months")

In [ ]:
# Part 6c — compute the metrics (after modis_monthly.csv is on Drive)
modis_table = pd.read_csv(f"{EXPORT_DIR}/modis_monthly.csv", parse_dates=["month"])
series = {key: g.set_index("month").sort_index() for key, g in modis_table.groupby(["point_id", "zone"])}

def lst(point_id, zone, band):
    return series[(point_id, zone)][band] if (point_id, zone) in series else pd.Series(dtype=float)

result_rows = []
for s in census_sites.itertuples():
    month, source = marinoni.operation_month(s.operation_start, years[s.site_id])
    placebo_ids = placebos.loc[placebos.site_id == s.site_id, "point_id"]
    for band in ("day", "night"):
        placebo = np.array([marinoni.marinoni_metric(lst(p, "disk", band), month) for p in placebo_ids], dtype=float)
        placebo_ref = np.array([marinoni.referenced_metric(lst(p, "disk", band), lst(p, "annulus", band), month)
                                for p in placebo_ids], dtype=float)
        result_rows.append({
            "site_id": s.site_id, "name": s.name, "group": s.group, "band": band,
            "op_month": month.date(), "op_source": source,
            "marinoni": marinoni.marinoni_metric(lst(s.site_id, "disk", band), month),
            "referenced": marinoni.referenced_metric(lst(s.site_id, "disk", band), lst(s.site_id, "annulus", band), month),
            "placebo_mean": np.nanmean(placebo) if np.isfinite(placebo).any() else np.nan,
            "placebo_sd": np.nanstd(placebo) if np.isfinite(placebo).any() else np.nan,
            "placebo_ref_mean": np.nanmean(placebo_ref) if np.isfinite(placebo_ref).any() else np.nan,
            "n_placebo": int(np.isfinite(placebo).sum())})
marinoni_results = pd.DataFrame(result_rows)
marinoni_results.to_csv(f"{OUT}/marinoni_results.csv", index=False)

for band in ("day", "night"):
    r = marinoni_results[marinoni_results.band == band]
    print(f"\n{band.upper()}  (mean over {r.marinoni.notna().sum()} sites)")
    print(f"  Marinoni metric at data centers   {r.marinoni.mean():+.2f} °C   ({(r.marinoni > 0).mean():.0%} positive)")
    print(f"  same metric at placebo points     {r.placebo_mean.mean():+.2f} °C")
    print(f"  background-referenced at sites    {r.referenced.mean():+.2f} °C   ({(r.referenced > 0).mean():.0%} positive)")
    print(f"  background-referenced at placebos {r.placebo_ref_mean.mean():+.2f} °C")
cols = ["name", "group", "op_source", "marinoni", "placebo_mean", "placebo_sd", "referenced", "n_placebo"]
print("\nDAY, by site:")
print(marinoni_results[marinoni_results.band == "day"].sort_values("marinoni")[cols]
      .to_string(index=False, float_format=lambda v: f"{v:+.2f}"))

### Part 6d–6e — Aqua re-run (orbit-drift check)

Terra's overpass has drifted earlier since 2020, which by itself cools daytime and warms night-time LST
in recent years. Aqua (MYD11A1) drifts differently, so if the placebo values change while the
background-referenced values stay near zero, the raw Marinoni metric is partly a sensor artifact.

In [ ]:
# Part 6d — start the Aqua export (self-contained; needs 6a run in this session)
importlib.reload(dc_heat.marinoni)
from dc_heat import marinoni
EXPORT_DIR = f"{OUT}/dc_heat_census_exports"          # the folder now lives under census/outputs
START_M, N_MONTHS, TASK_AQUA = "2004-01-01", 272, "modis_monthly_aqua"
AQUA_PATHS = [f"{EXPORT_DIR}/{TASK_AQUA}.csv", f"/content/drive/MyDrive/{EXPORT_FOLDER}/{TASK_AQUA}.csv"]

points = ([(s.site_id, s.site_id, "site", s.lon, s.lat) for s in census_sites.itertuples()] +
          [(p.site_id, p.point_id, "placebo", p.lon, p.lat) for p in placebos.itertuples()])
features = []
for site_id, point_id, kind, lon, lat in points:
    pt = ee.Geometry.Point(lon, lat)
    props = {"site_id": site_id, "point_id": point_id, "kind": kind}
    features.append(ee.Feature(pt.buffer(10000, 100), {**props, "zone": "disk"}))
    features.append(ee.Feature(pt.buffer(20000, 100).difference(pt.buffer(10000, 100), 100), {**props, "zone": "annulus"}))
zones = ee.FeatureCollection(features)

aqua = ee.ImageCollection("MODIS/061/MYD11A1").select(["LST_Day_1km", "LST_Night_1km"])
def monthly_aqua(i):
    start = ee.Date(START_M).advance(i, "month")
    img = aqua.filterDate(start, start.advance(1, "month")).mean().multiply(0.02).subtract(273.15)
    return img.rename(["day", "night"]).set("month", start.format("YYYY-MM"))
def per_month(img):
    return (img.reduceRegions(collection=zones, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
               .map(lambda f: f.set("month", img.get("month"))))
table_aqua = ee.ImageCollection.fromImages(ee.List.sequence(0, N_MONTHS - 1).map(monthly_aqua)).map(per_month).flatten()

existing = [t for t in ee.batch.Task.list() if t.config["description"] == TASK_AQUA]
if any(os.path.exists(p) for p in AQUA_PATHS):
    print(f"{TASK_AQUA}.csv is on Drive — run 6e")
elif existing and existing[0].state in ("READY", "RUNNING", "COMPLETED"):
    print(f"{TASK_AQUA}: {existing[0].state}")
else:
    ee.batch.Export.table.toDrive(collection=table_aqua, description=TASK_AQUA, folder=EXPORT_FOLDER,
                                  fileNamePrefix=TASK_AQUA, fileFormat="CSV",
                                  selectors=["site_id", "point_id", "kind", "zone", "month", "day", "night"]).start()
    print(f"started {TASK_AQUA}: {len(points)} points x 2 zones x {N_MONTHS} months")

In [ ]:
# Part 6e — Terra vs Aqua (after the Aqua export finishes)
aqua_path = next(p for p in AQUA_PATHS if os.path.exists(p))
aqua_results = marinoni.metric_table(pd.read_csv(aqua_path), census_sites, placebos, years)
aqua_results.to_csv(f"{OUT}/marinoni_results_aqua.csv", index=False)
terra_results = pd.read_csv(f"{OUT}/marinoni_results.csv")
print(f"Aqua file read from: {aqua_path}\n")
print(f"{'':12s}{'sites':>9s}{'placebos':>10s}{'referenced':>12s}{'placebo-ref':>13s}")
for band in ("day", "night"):
    for sensor, r in (("Terra", terra_results), ("Aqua", aqua_results)):
        d = r[r.band == band]
        print(f"{band:5s} {sensor:6s}{d.marinoni.mean():+9.2f}{d.placebo_mean.mean():+10.2f}"
              f"{d.referenced.mean():+12.2f}{d.placebo_ref_mean.mean():+13.2f}")